In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Select Device According to Availability
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("Device selected:", device)

Device selected: cuda


In [3]:
class Window:
    def __init__(self, video_path, window_size=4, stride=2, frame_average_count=5):
        """
        A class to manage sliding windows over video frames.

        :param video_path (str): The path to the video file.
        :param window_size (int): The size of the sliding window (default is 4).
        :param stride (int): The step size for moving the window (default is 2).
        :param frame_average_count (int): The number of frames to be averaged for each group (default is 5).
        """
        # The duration of the video in seconds
        self.__video_duration = None
        # The size of the sliding window
        self.__window_size = window_size
        # The step size for moving the window
        self.__stride = stride
        # The path to the video file
        self.__video_path = video_path
        # An array containing processed frames from the video
        self.__total_frames = self.__get_frames()
        # A sliding window of frames
        self.__windows = self.__prepare_window()
        # The index of the current window in the windowed clip
        self.__window_index = 0

    def next(self):
        """
        This method strides to the next window in the windowed clip.
        :return: numpy.ndarray: The next window in the windowed clip.
        """
        if self.has_next():
            self.__window_index += 1
            return self.__windows[self.__window_index - 1]
        else:
            print("No next window")
            return None

    def has_next(self):
        """
        This method checks if there is a next window in the windowed clip.
        :return: bool: True if there is a next window, False otherwise.
        """
        return self.__window_index < len(self.__windows)

    def current_window(self):
        """
        This method returns the current window in the windowed clip.
        :return: numpy.ndarray: The current window in the windowed clip.
        """
        return self.__windows[self.__window_index]

    def previous(self):
        """
        This method strides to the previous window in the windowed clip.
        :return: numpy.ndarray: The previous window in the windowed clip.
        """
        if self.__window_index > 0:
            self.__window_index -= 1
            return self.__windows[self.__window_index]
        else:
            print("No previous window")
            return None

    def reset(self):
        """
        This method resets the window index to the beginning of the windowed clip.
        """
        self.__window_index = 0

    def __prepare_window(self):
        """
        Prepares a sliding window of frames from the average frame of the video.

        :returns:
            numpy.ndarray: A sliding window of frames.

        Notes:
            This method divides the average frame into overlapping windows of 'window_size' frames.
            The stride parameter determines the step size for moving the window.
            The number of steps is calculated based on the average frame shape and video duration.
            The resulting windows are stored in a numpy array.
        """
        window = []
        fps = self.__total_frames.shape[0] // self.__video_duration
        steps = int(round((fps * self.__window_size) // self.__stride))

        for i in range(0, len(self.__total_frames) - steps, steps):
            window.append(self.__total_frames[i: i + steps * 2])
        return np.array(window)

    def __get_frames(self):
        """
        This method reads the video file and returns the frames

        :return:
            numpy.ndarray: An array containing processed frames from the video.
        :raises:
            IOError: If the video file cannot be read or does not exist.
        """
        video = cv2.VideoCapture(self.__video_path)
        if not video.isOpened():
            raise IOError("Error reading video file")

        frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = video.get(cv2.CAP_PROP_FPS)
        self.__video_duration = frame_count // fps  # Adjust if needed for partial frames

        frames = []
        for i in range(0, frame_count, 3):
            video.set(1, i)
            ret, frame = video.read()
            if ret:
                frame = cv2.resize(frame, (224, 224))
                frame = frame.astype("float32") / 255.0
                frames.append(frame)
        video.release()
        return np.array(frames)

    def __iter__(self):
        """
        This method makes the object iterable.
        :return: window: The object itself.
        """
        return self  # Return self to make the object iterable

    def __next__(self):
        """
        This method returns the next window in the windowed clip. (used for iteration)
        :return: numpy.ndarray: The next window in the windowed clip.
        """
        if self.has_next():
            self.__window_index += 1
            return self.__windows[self.__window_index - 1]
        else:
            raise StopIteration

    def get_current_window_stats(self):
        """
        This method returns the stats of the current window.

        :return:
            dict: A dictionary containing the statistics of the current window.
        """
        return {
            "index": self.__window_index,
            "frame_count": len(self.__windows[self.__window_index]),
            "frame_shape": self.__windows[self.__window_index].shape,
            "frame_dtype": self.__windows[self.__window_index].dtype,
        }

    def __repr__(self):
        return (f"Window (\n\tVideo Path = {self.__video_path}\n"
                f"\tTotal Window = {self.__window_size}\n"
                f"\tStride = {self.__stride}\n"
                f"\tNo. Frames Averaged = {self.__frame_average_count}\n"
                f"\tTotal Frames = {len(self.__total_frames)}\n"
                f"\tAveraged Frames = {len(self.__averaged_frames)}\n"
                f"\tShape = {self.__windows.shape}\n)")

    @property
    def shape(self):
        return self.__windows.shape


In [4]:
class WindowEmbedded:
    def __init__(self, windows: Window):
        self.__windows = windows
        self.__embedding_model = CNN().to(device)
        self.window_embeddings = self.__get_embeddings()

    # rest of the code remains the same
    def __get_embeddings(self):
        embeddings_list = []
        # for window in tqdm.tqdm(self.__windows, desc="Extracting Embeddings", total=self.__windows.shape[0]):
        for window in self.__windows:
            frames = torch.tensor(window).permute(0, 3, 1, 2).to(device)
            frame_embeddings_list = []
            for frame in frames:
                frame = frame.unsqueeze(0)
                frame = frame.to(device)
                frame_embeddings = self.__embedding_model(frame)
                frame_embeddings_list.append(frame_embeddings.flatten().cpu().detach().numpy())
            embeddings_list.append(frame_embeddings_list)
        self.window_embeddings = np.array(embeddings_list)
        return self.window_embeddings

In [5]:
class CNN(nn.Module):
    """
    A simple convolutional neural network (CNN) for generating embeddings from frames.

    Methods:
        forward(self, x): Defines the forward pass of the model.
    """
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 0, device=device)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 0, device=device)
        self.d = nn.Dropout(0.5)
        self.conv3 = nn.Conv2d(64, 128, 3, 1, 0, device=device)
        # Pooling layer, All are same
        self.pool = nn.MaxPool2d(2, 2)
        # Fully connected layer
        self.fc1 = nn.Linear(128 * 26 * 26, 1024, device=device)  # Adjust input size based on your frame size

    def forward(self, x):
        """
        Forward pass of the CNN.

        Takes an input batch of images `x` and performs the following:
            1. Converts the input to the device (e.g., GPU).
            2. Applies ReLU activation to the output of the first convolutional layer.
            3. Performs max pooling with a kernel size of 2.
            4. Repeats steps 2 and 3 for the second and third convolutional layers.
            5. Flattens the output of the last pooling layer.
            6. Applies ReLU activation to the output of the fully-connected layer.
            7. Returns the final feature vector.

        Args:
            x (torch.Tensor): Input batch of images of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Output feature vector of shape (batch_size, 1024).
        """
        x = x.to(device)
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        # For batch size 1
        x = x.view(-1, 128 * 26 * 26)
        # For batch size > 1
        # x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return x

In [6]:
from torchvision import models
import torch.nn as nn

class ResNetAnomaly(nn.Module):
    def __init__(self):
        
        super(ResNetAnomaly, self).__init__()

        # Load the pre-trained ResNet50 model without fine-tuning
        self.resnet = models.resnet50(pretrained=False)
        self.resnet.load_state_dict(torch.load('/kaggle/input/resnet50/resnet50.pth'))

        # Get the number of features from the last layer of the ResNet
        num_ftrs = self.resnet.fc.in_features

        # Modify the last layer to output 2048 features
        self.resnet.fc = nn.Linear(num_ftrs, 2048)

        # Define additional layers for feature learning
        self.additional_layers = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = x.to(device)
        x = self.resnet(x)
        # Pass the features through the additional layers for further learning
        x = self.additional_layers(x)

        return x

In [7]:
import numpy as np
import glob
from random import shuffle
import tqdm

train_path = '/kaggle/input/augmenteddataset/train/*/*.mp4'
train_paths = glob.glob(train_path)
train_labels = [1 if i.split('/')[5] == 'anomalous' else 0 for i in train_paths]

train_zip = list(zip(train_paths, train_labels))
shuffle(train_zip)
train_paths, train_labels = zip(*train_zip)

train_labels = list(train_labels)

In [8]:
def get_all_video_embeddings(device):
    valid_labels = []
    all_video_embedding = []

    for video_path, label in zip(tqdm.tqdm(train_paths), train_labels):
        try:
            window = Window(video_path)
            window_embed_object = WindowEmbedded(window)
            embeddings = window_embed_object.window_embeddings
            all_video_embedding.append(embeddings)
            valid_labels.append(label)
        except ValueError as e:
            print(f"Skipping video {video_path} due to error: {e}")
            continue
    
    return all_video_embedding, valid_labels

In [9]:
all_video_embedding, valid_labels = get_all_video_embeddings(device)


from keras.preprocessing.sequence import pad_sequences

# Calculate the maximum sequence length
#total_window, total_frames, total_embedding, 4, 24, 1024
max_sequence_length = max(window.shape[0] for video_embedding in all_video_embedding for window in video_embedding)

all_video_embedding_padded = []

for video_embedding in all_video_embedding:
    video_embedding_padded = []
    for window in video_embedding:
        if window.shape[0] != max_sequence_length:
            # Apply padding only if the window shape does not match the max_sequence_length
            window_padded = pad_sequences([window], maxlen=max_sequence_length, padding='post', dtype='float32')[0]
            video_embedding_padded.append(window_padded)
        else:
            video_embedding_padded.append(window)
    all_video_embedding_padded.append(video_embedding_padded)

all_video_embedding_padded = np.array(all_video_embedding_padded)


np.save('/kaggle/working/train_embeddings.npy', all_video_embedding_padded) 
np.save('/kaggle/working/train_labels.npy', np.array(valid_labels))

  0%|          | 26/9150 [00:18<1:27:49,  1.73it/s]

Skipping video /kaggle/input/augmenteddataset/train/nonanomalous/Normal_Videos_014_x264_004.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


  0%|          | 35/9150 [00:23<1:25:43,  1.77it/s]

Skipping video /kaggle/input/augmenteddataset/train/anomalous/865_p3_Arson031_x264_003_flipped.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


  1%|          | 113/9150 [01:16<1:28:36,  1.70it/s]

...

 97%|█████████▋| 8862/9150 [1:39:49<02:52,  1.67it/s]

Skipping video /kaggle/input/augmenteddataset/train/anomalous/721_p2_Arson011_x264_000_flipped.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


 98%|█████████▊| 8987/9150 [1:41:13<01:34,  1.72it/s]

Skipping video /kaggle/input/augmenteddataset/train/nonanomalous/Burglary038_x264_002.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


100%|██████████| 9150/9150 [1:43:04<00:00,  1.48it/s]
2024-03-15 06:52:39.817340: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-15 06:52:39.817427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-15 06:52:39.953825: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
